In [1]:
# Cell 1: Imports
import cv2
import numpy as np
from pathlib import Path
import shutil
from tqdm import tqdm
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent
TRAIN_DIR = PROJECT_ROOT / 'data' / 'splits' / 'train'

print('Project root:', PROJECT_ROOT)
print('Train dir:', TRAIN_DIR)
print('Train dir exists:', TRAIN_DIR.exists())

Project root: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification
Train dir: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification\data\splits\train
Train dir exists: True


In [2]:
# Cell 2: Analyze class distribution
class_folders = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir() and d.name.startswith('class_')])

class_stats = []
for class_dir in class_folders:
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
    class_stats.append({
        'class_name': class_dir.name,
        'class_path': class_dir,
        'count': len(images),
        'images': images
    })

print('Class Distribution (Training Set):')
print('='*70)
for stat in class_stats:
    print(f"{stat['class_name']:40s}: {stat['count']:4d} images")

print('\n' + '='*70)
total_images = sum(s['count'] for s in class_stats)
print(f'Total training images: {total_images}')
print(f'Min class size: {min(s["count"] for s in class_stats)}')
print(f'Max class size: {max(s["count"] for s in class_stats)}')

Class Distribution (Training Set):
class_01_motorcycle                     :  198 images
class_02_passenger_car                  :  523 images
class_03_four_tire_single_unit          :   47 images
class_04_bus                            :  182 images
class_05_two_axle_six_tire_single_unit  :   55 images
class_06_three_axle_single_unit         :   82 images
class_07_four_or_more_axle_single_unit  :   28 images
class_08_four_or_less_axle_single_trailer:   41 images
class_09_five_axle_tractor_semitrailer  :   48 images
class_10_six_or_more_axle_single_trailer:   39 images
class_11_five_or_less_axle_multi_trailer:   28 images
class_13_seven_or_more_axle_multi_trailer:   39 images

Total training images: 1310
Min class size: 28
Max class size: 523


In [3]:
# Cell 3: Determine augmentation strategy
def calculate_augmentation_factor(count):
    """Determine how many augmented versions to create per image."""
    if count < 50:
        # Target ~400 samples for very small classes
        return max(1, min(8, 400 // count))
    elif count < 100:
        # Target ~250 samples for small classes
        return max(1, min(3, 250 // count))
    else:
        # No augmentation for well-represented classes
        return 0

print('Augmentation Plan:')
print('='*70)
for stat in class_stats:
    aug_factor = calculate_augmentation_factor(stat['count'])
    new_total = stat['count'] + (stat['count'] * aug_factor)
    stat['aug_factor'] = aug_factor
    stat['target_count'] = new_total
    
    if aug_factor > 0:
        print(f"{stat['class_name']:40s}: {stat['count']:4d} → {new_total:4d} ({aug_factor}x augmentation)")
    else:
        print(f"{stat['class_name']:40s}: {stat['count']:4d} (no augmentation)")

print('\n' + '='*70)
total_new = sum(s['target_count'] for s in class_stats)
total_augmented = total_new - total_images
print(f'Total after augmentation: {total_new} (+{total_augmented} augmented images)')

Augmentation Plan:
class_01_motorcycle                     :  198 (no augmentation)
class_02_passenger_car                  :  523 (no augmentation)
class_03_four_tire_single_unit          :   47 →  423 (8x augmentation)
class_04_bus                            :  182 (no augmentation)
class_05_two_axle_six_tire_single_unit  :   55 →  220 (3x augmentation)
class_06_three_axle_single_unit         :   82 →  328 (3x augmentation)
class_07_four_or_more_axle_single_unit  :   28 →  252 (8x augmentation)
class_08_four_or_less_axle_single_trailer:   41 →  369 (8x augmentation)
class_09_five_axle_tractor_semitrailer  :   48 →  432 (8x augmentation)
class_10_six_or_more_axle_single_trailer:   39 →  351 (8x augmentation)
class_11_five_or_less_axle_multi_trailer:   28 →  252 (8x augmentation)
class_13_seven_or_more_axle_multi_trailer:   39 →  351 (8x augmentation)

Total after augmentation: 3881 (+2571 augmented images)


In [4]:
# Cell 4: Define augmentation functions
def augment_horizontal_flip(img):
    """Horizontal flip - very realistic for vehicles (left/right lane)."""
    return cv2.flip(img, 1)

def augment_rotate(img, angle):
    """Rotate image by specified angle (in degrees)."""
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def augment_brightness(img, factor):
    """Adjust brightness by factor (0.8 = darker, 1.2 = brighter)."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def augment_contrast(img, alpha):
    """Adjust contrast (alpha > 1 increases contrast)."""
    return cv2.convertScaleAbs(img, alpha=alpha, beta=0)

# Define augmentation pipeline variants
AUGMENTATION_VARIANTS = [
    ('flip', lambda img: augment_horizontal_flip(img)),
    ('rot_10', lambda img: augment_rotate(img, 10)),
    ('rot_m10', lambda img: augment_rotate(img, -10)),
    ('bright_1.2', lambda img: augment_brightness(img, 1.2)),
    ('bright_0.8', lambda img: augment_brightness(img, 0.8)),
    ('flip_bright', lambda img: augment_brightness(augment_horizontal_flip(img), 1.2)),
    ('flip_rot', lambda img: augment_rotate(augment_horizontal_flip(img), 10)),
    ('contrast_1.2', lambda img: augment_contrast(img, 1.2)),
]

print(f'Defined {len(AUGMENTATION_VARIANTS)} augmentation variants:')
for name, _ in AUGMENTATION_VARIANTS:
    print(f'  - {name}')

Defined 8 augmentation variants:
  - flip
  - rot_10
  - rot_m10
  - bright_1.2
  - bright_0.8
  - flip_bright
  - flip_rot
  - contrast_1.2


In [5]:
# Cell 4b: Create backup of training data (RECOMMENDED - run this before augmentation!)
import datetime

BACKUP_DIR = PROJECT_ROOT / 'data' / 'splits' / 'train_backup'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP_DIR_TIMESTAMPED = PROJECT_ROOT / 'data' / 'splits' / f'train_backup_{timestamp}'

print('Creating backup of training data...')
print('='*70)
print(f'Source: {TRAIN_DIR}')
print(f'Backup: {BACKUP_DIR_TIMESTAMPED}')
print()

# Check if backup already exists
if BACKUP_DIR.exists():
    print(f'⚠ WARNING: Backup already exists at {BACKUP_DIR}')
    print('  Previous backup will not be overwritten. Creating timestamped backup instead.')

# Create timestamped backup
try:
    shutil.copytree(TRAIN_DIR, BACKUP_DIR_TIMESTAMPED)
    
    # Count files
    backup_files = sum(len(list(d.glob('*.jpg')) + list(d.glob('*.png'))) 
                      for d in BACKUP_DIR_TIMESTAMPED.iterdir() if d.is_dir())
    
    print('✓ Backup created successfully!')
    print(f'  Location: {BACKUP_DIR_TIMESTAMPED}')
    print(f'  Files backed up: {backup_files}')
    print(f'  Disk space used: ~{sum(f.stat().st_size for f in BACKUP_DIR_TIMESTAMPED.rglob("*") if f.is_file()) / (1024**2):.1f} MB')
    
    print('\nYou can now safely run the augmentation cell.')
    print('To restore backup if needed: manually copy files from backup folder back to train/')
    
except Exception as e:
    print(f'❌ ERROR creating backup: {e}')
    print('RECOMMENDATION: Do not run augmentation until backup succeeds!')

Creating backup of training data...
Source: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification\data\splits\train
Backup: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification\data\splits\train_backup_20260113_161950

✓ Backup created successfully!
  Location: c:\Users\amrem\Documents\Engineering Year 4\Machine Learning\Project\Vehicle-Classification\data\splits\train_backup_20260113_161950
  Files backed up: 1310
  Disk space used: ~365.0 MB

You can now safely run the augmentation cell.
To restore backup if needed: manually copy files from backup folder back to train/


In [6]:
# Cell 5: Apply augmentation (RUN THIS TO AUGMENT IMAGES)
# WARNING: This will create new image files in your training directories!
# Make sure you have a backup if needed.

print('Starting augmentation...')
print('='*70)

total_augmented_count = 0

for stat in class_stats:
    aug_factor = stat['aug_factor']
    
    if aug_factor == 0:
        print(f"Skipping {stat['class_name']} (sufficient samples)")
        continue
    
    print(f"\nAugmenting {stat['class_name']} ({stat['count']} images, {aug_factor}x)...")
    
    class_dir = stat['class_path']
    images = stat['images']
    
    augmented_in_class = 0
    
    # For each original image, create aug_factor augmented versions
    for img_path in tqdm(images, desc=f"  {stat['class_name']}"):
        # Read image
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"    WARNING: Could not read {img_path.name}")
            continue
        
        # Select aug_factor random augmentation variants
        selected_variants = random.sample(AUGMENTATION_VARIANTS, min(aug_factor, len(AUGMENTATION_VARIANTS)))
        
        for variant_name, aug_func in selected_variants:
            try:
                # Apply augmentation
                aug_img = aug_func(img)
                
                # Generate output filename: original_name_aug_variant.jpg
                stem = img_path.stem
                ext = img_path.suffix
                out_name = f"{stem}_aug_{variant_name}{ext}"
                out_path = class_dir / out_name
                
                # Save augmented image
                cv2.imwrite(str(out_path), aug_img)
                augmented_in_class += 1
                
            except Exception as e:
                print(f"    ERROR augmenting {img_path.name} with {variant_name}: {e}")
    
    total_augmented_count += augmented_in_class
    print(f"  Created {augmented_in_class} augmented images for {stat['class_name']}")

print('\n' + '='*70)
print(f'✓ Augmentation complete!')
print(f'  Total augmented images created: {total_augmented_count}')
print(f'  Total training images now: {total_images + total_augmented_count}')

Starting augmentation...
Skipping class_01_motorcycle (sufficient samples)
Skipping class_02_passenger_car (sufficient samples)

Augmenting class_03_four_tire_single_unit (47 images, 8x)...


  class_03_four_tire_single_unit: 100%|██████████| 47/47 [00:02<00:00, 17.46it/s] 


  Created 376 augmented images for class_03_four_tire_single_unit
Skipping class_04_bus (sufficient samples)

Augmenting class_05_two_axle_six_tire_single_unit (55 images, 3x)...


  class_05_two_axle_six_tire_single_unit: 100%|██████████| 55/55 [00:01<00:00, 50.37it/s]


  Created 165 augmented images for class_05_two_axle_six_tire_single_unit

Augmenting class_06_three_axle_single_unit (82 images, 3x)...


...

  Created 312 augmented images for class_10_six_or_more_axle_single_trailer

Augmenting class_11_five_or_less_axle_multi_trailer (28 images, 8x)...


  class_11_five_or_less_axle_multi_trailer: 100%|██████████| 28/28 [00:01<00:00, 14.13it/s]


  Created 224 augmented images for class_11_five_or_less_axle_multi_trailer

Augmenting class_13_seven_or_more_axle_multi_trailer (39 images, 8x)...


  class_13_seven_or_more_axle_multi_trailer: 100%|██████████| 39/39 [00:04<00:00,  8.30it/s]

  Created 312 augmented images for class_13_seven_or_more_axle_multi_trailer

✓ Augmentation complete!
  Total augmented images created: 2571
  Total training images now: 3881


In [7]:
# Cell 6: Verify new class distribution
print('Verifying new class distribution...')
print('='*70)

new_total = 0
for class_dir in class_folders:
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
    new_count = len(images)
    new_total += new_count
    
    # Find original count
    orig_count = next(s['count'] for s in class_stats if s['class_path'] == class_dir)
    diff = new_count - orig_count
    
    print(f"{class_dir.name:40s}: {orig_count:4d} → {new_count:4d} (+{diff})")

print('\n' + '='*70)
print(f'Total: {total_images} → {new_total} (+{new_total - total_images})')
print(f'\nNew min class size: {min(len(list(d.glob("*.jpg")) + list(d.glob("*.png"))) for d in class_folders)}')
print(f'New max class size: {max(len(list(d.glob("*.jpg")) + list(d.glob("*.png"))) for d in class_folders)}')

Verifying new class distribution...
class_01_motorcycle                     :  198 →  198 (+0)
class_02_passenger_car                  :  523 →  523 (+0)
class_03_four_tire_single_unit          :   47 →  423 (+376)
class_04_bus                            :  182 →  182 (+0)
class_05_two_axle_six_tire_single_unit  :   55 →  220 (+165)
class_06_three_axle_single_unit         :   82 →  328 (+246)
class_07_four_or_more_axle_single_unit  :   28 →  252 (+224)
class_08_four_or_less_axle_single_trailer:   41 →  369 (+328)
class_09_five_axle_tractor_semitrailer  :   48 →  432 (+384)
class_10_six_or_more_axle_single_trailer:   39 →  351 (+312)
class_11_five_or_less_axle_multi_trailer:   28 →  252 (+224)
class_13_seven_or_more_axle_multi_trailer:   39 →  351 (+312)

Total: 1310 → 3881 (+2571)

New min class size: 182
New max class size: 523


## Next Steps

After running the augmentation:

1. **Re-run feature extraction** (`2_feature_selection.ipynb`)
   - This will extract features from both original + augmented images
   - Generate new `features_train_pca.csv` etc. with balanced classes

2. **Re-run model training** (`3_traditional_ml_training.ipynb`)
   - Train models on the new balanced feature set
   - Should see improved F1-macro and minority class recall

3. **Compare results**
   - Before augmentation: ~68% test accuracy, ~43% F1-macro
   - After augmentation: expect similar accuracy but better F1-macro

**Note:** If you want to undo augmentation, simply delete all files with `_aug_` in their names.

In [8]:
# Cell 7: Remove all augmented images (if you want to undo)
# Run this cell to delete all augmented images

print('Removing augmented images...')
removed_count = 0
for class_dir in class_folders:
    aug_images = list(class_dir.glob('*_aug_*'))
    for img_path in aug_images:
        img_path.unlink()
        removed_count += 1
print(f'Removed {removed_count} augmented images')

Removing augmented images...
Removed 2571 augmented images
